# Multi‑lead evaluation demo

This notebook demonstrates the new multi‑lead features: per‑lead histograms, map panels, KDE evolution, energy spectrograms, deterministic/ETS/probabilistic line plots, and vertical profile evolution, using small synthetic data for speed.

In [ ]:
# Imports
import numpy as np
import xarray as xr
from pathlib import Path
from swissclim_evaluations.cli import run_selected

In [ ]:
# Build tiny synthetic datasets with init_time x lead_time x lat x lon
n_init = 2; n_lead = 6; ny = 32; nx = 64
init_time = np.array(['2023-01-01T00','2023-01-02T00'], dtype='datetime64[h]')
lead_time = np.array([i for i in range(n_lead)], dtype='timedelta64[h]')
lat = np.linspace(-89.5, 89.5, ny)
lon = np.linspace(0.0, 360.0 - 360.0/nx, nx)
def synth_field(seed=0):
    rng = np.random.default_rng(seed)
    base = rng.standard_normal((n_init, n_lead, ny, nx)).astype('float32')
    # Add simple lead_time trend and a zonal wave
    kx = 4
    xwave = np.sin(2*np.pi * (np.arange(nx)[None,None,None,:] * kx / nx)).astype('float32')
    trend = (np.arange(n_lead, dtype='float32')[None,:,None,None]) / n_lead
    return base + 0.5 * xwave + trend
tgt = xr.Dataset({
# Prediction with slight bias and noise
pred = xr.Dataset({

In [ ]:
# Write synthetic datasets to temporary Zarr stores (in-memory stores could also be used)
tmp_root = Path('notebooks/.tmp_demo')
tmp_root.mkdir(parents=True, exist_ok=True)
ml_store = tmp_root / 'pred.zarr'
nwp_store = tmp_root / 'tgt.zarr'
pred.to_zarr(ml_store, mode='w')
tgt.to_zarr(nwp_store, mode='w')
ml_store, nwp_store

In [ ]:
# Build an in-memory config enabling multi-lead visualizations
cfg = {
# The CLI expects a 'lead_time' policy — use full for demo
cfg['lead_time'] = {'mode': 'full'}
cfg['_config_path'] = 'in-memory'  # for provenance copy no-op
cfg

In [ ]:
# Run the pipeline on synthetic data (fast)
run_selected(cfg)